# Inference in RAG

RAG - Retrieval-Augmented Generation (Generación aumentada con recuperación). Es una técnica dentro de la inteligencia artificial, sobre todo aplicada en modelos de lenguaje como los que se usan en chatbots o asistentes.

## Cómo funciona

	•	Recuperación (Retrieval): el sistema busca información relevante en una base de datos, documentos o fuentes externas.
    
	•	Generación (Generation): el modelo de lenguaje usa esa información recuperada para generar una respuesta más precisa, contextualizada y actualizada.

En otras palabras, el modelo no se limita solo a lo que ya aprendió durante su entrenamiento, sino que consulta información externa en tiempo real y la integra en su respuesta.

**Ejemplo práctico**

Si preguntas a un modelo con RAG: “¿Qué dice el informe de la ONU de 2024 sobre cambio climático?”, primero recuperará el texto del informe y luego generará una respuesta basada en ese contenido, en lugar de inventar o usar solo datos antiguos.

Ventajas:

	•	Respuestas más precisas y actualizadas.
    
	•	Reduce la alucinación (cuando un modelo inventa datos).
    
	•	Permite integrar bases de conocimiento privadas o especializadas (documentos internos, artículos científicos, manuales, etc.).


In [ ]:
#
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, \
                         DPRContextEncoder, DPRQuestionEncoder

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#
def cosine_similarity_matrix(features):
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    normalized_features = features / norms
    similarity_matrix = np.inner(normalized_features, normalized_features)
    rounded_similarity_matrix = np.round(similarity_matrix, 4)
    return rounded_similarity_matrix

Este notebook cubre solo la parte de **recuperación** de RAG (los pasos 1 y 2 de la ecuación "Recuperación (Retrieval) + Generación" descrita arriba); no incluye el paso 3 (pasarle los documentos recuperados a un LLM para que genere una respuesta) — eso es exactamente lo que hacen los notebooks de agentes (`04 Modelos de IA Generativa`), que sí combinan recuperación + generación.

Aquí comparamos dos maneras de implementar la recuperación:

**1. Similitud "pura" (un solo modelo de embeddings).** La forma más simple: usamos el *mismo* modelo de embeddings para la pregunta y para cada candidata a respuesta, y elegimos la candidata cuyo embedding esté más cerca (similitud coseno) del embedding de la pregunta.

## Pure similarity

In [ ]:
#
answers = [
    "What is the tallest mountain in the world?",
    "The tallest mountain in the world is Mount Everest.",
    "Mount Shasta",
    "I like my hike in the mountains",
    "I am going to a yoga class"
]

question = 'What is the tallest mountain in the world?'

In [ ]:
#
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
question_embedding = list(model.encode(question))

In [ ]:
#
sim = []
for answer in answers:
    answer_embedding = list(model.encode(answer))
    sim.append(cosine_similarity_matrix(np.stack([question_embedding, answer_embedding]))[0,1])

print(sim)
best_inx = np.argmax(sim)
print(f"Question = {question}")
print(f"Best answer = {answers[best_inx]}")

**2. Dual-Encoder (DPR — Dense Passage Retrieval).** La similitud "pura" tiene una limitación: una pregunta y su respuesta correcta no siempre se parecen léxicamente ("¿Cuál es la montaña más alta del mundo?" vs. "El Monte Everest es la montaña más alta del mundo"), así que un solo modelo de embeddings entrenado para similitud simétrica (oración A se parece a oración B) no siempre es el más adecuado para esta tarea *asimétrica* (pregunta busca respuesta).

DPR resuelve esto con **dos encoders separados**, entrenados juntos pero especializados cada uno en su rol: un `question_encoder` (para preguntas) y un `context_encoder` (para los pasajes/documentos candidatos). Al entrenarse específicamente para que preguntas y sus respuestas correctas queden cerca en el espacio vectorial —aunque no compartan las mismas palabras—, un dual-encoder entrenado para recuperación suele superar a la similitud "pura" en tareas de pregunta-respuesta.

## Dual-Encoder inference

In [ ]:
#
answer_tokenizer = AutoTokenizer \
                   .from_pretrained("facebook/dpr-ctx_encoder-multiset-base")
answer_encoder = DPRContextEncoder \
                   .from_pretrained("facebook/dpr-ctx_encoder-multiset-base")

question_tokenizer = AutoTokenizer \
                   .from_pretrained("facebook/dpr-question_encoder-multiset-base")
question_encoder = DPRQuestionEncoder \
                   .from_pretrained("facebook/dpr-question_encoder-multiset-base")

In [ ]:
# Compute the question embeddings
question_tokens = question_tokenizer(question, return_tensors="pt")["input_ids"]
question_embedding = question_encoder(question_tokens).pooler_output.flatten().tolist()


In [ ]:
#
print(question_embedding[:10], len(question_embedding))

In [ ]:
#
sim = []
for answer in answers:
    answer_tokens = answer_tokenizer(answer, return_tensors="pt")["input_ids"]
    answer_embedding = answer_encoder(answer_tokens).pooler_output.flatten().tolist()
    sim.append(cosine_similarity_matrix(np.stack([question_embedding, answer_embedding]))[0,1])


In [ ]:
#
print(sim)
best_inx = np.argmax(sim)
print(f"Question = {question}")
print(f"Best answer = {answers[best_inx]}")

## Para pensar

1. Compara los resultados de las dos secciones (`sim` en "Pure similarity" vs. `sim` en "Dual-Encoder inference") sobre las mismas `answers`. ¿Ambos enfoques eligen la misma `best_answer`? Si difieren, ¿cuál te parece más razonable dado que la pregunta y la respuesta correcta comparten pocas palabras literales?

2. Los "distractores" de la lista `answers` (`"Mount Shasta"`, `"I like my hike in the mountains"`, `"I am going to a yoga class"`) están ordenados de más a menos relacionados con la pregunta. ¿Los puntajes de similitud (`sim`) respetan ese orden en ambos métodos? Si no, ¿qué te dice eso sobre las limitaciones de medir "relevancia" solo con similitud vectorial?

3. **Ejercicio:** agrega dos o tres candidatas a respuesta nuevas (relacionadas con otro tema, no montañas) a la lista `answers`, y agrega una pregunta nueva. Corre ambos métodos de nuevo — ¿siguen recuperando la respuesta correcta?